In [1]:
import os
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
plt.rcParams["figure.figsize"] = (24,18)
from utils import *

In [2]:
model_path = os.path.join(os.path.dirname(os.getcwd()), 'models')
model = YOLO(os.path.join(model_path, "yolov8n.pt"))
data_path = os.path.join(os.path.dirname(os.getcwd()), 'data')
vid_name =  'real_test'

In [3]:
# Get several images to test
images_path = os.path.join(data_path, 'images' + '/' + vid_name)  #print(os.path.exists(images_path))
destination_path_coco = os.path.join(data_path, 'coco_dataset' +'/'+  vid_name)

# Initialize fixed COCO attributes
coco_dict ={}
frame_id = 1
obj_id = 1
categories_dict = [{"id" : 0, "name" : "ball", "supercategory" : None},
                   {"id" : 1, "name" : "player_team_1", "supercategory" : None},
                   {"id" : 2, "name" : "player_team_2", "supercategory" : None},
                   {"id" : 3, "name" : "keeper_team_1", "supercategory" : None},
                   {"id" : 4, "name" : "keeper_team_2", "supercategory" : None},
                   {"id" : 5, "name" : "referee", "supercategory" : None},
                   {"id" : 6, "name" : "misc", "supercategory" : None}]

creation_date = date.today()
year = creation_date.year
coco_info = {
    'contributor': 'Khoa Nguyen, Huy Nguyen',
    'description': vid_name,
    'url': '',
    'version': 0,
    'date_created': str(creation_date),
    'year': year,
}

# Attributes to identify colors
global_color_dict = {'team_1_color' : [],
                     'team_1_position' : [],
                     'team_2_color' : [],
                     'team_2_position' : [],
                     'misc_color' : [],
                     'misc_position' : [],
                     'misc_frequency' : [],
                     'misc_box_coord' : []}


# Saved info to be write COCO file:
all_image_info = []
all_bbox_info = []

frame_num = 0

#first_batch = os.listdir(images_path)[0:20]
#mage_batch = [os.path.join(images_path, x) for x in first_batch]

for image_name in os.listdir(images_path):
    
    # Crop out the audiences via pitch segmentation
    image_ad  = os.path.join(images_path, image_name)
    rgb_image = cv2.imread(image_ad, cv2.COLOR_BGR2RGB)
    rgb_image = pitch_segmentation(rgb_image)
    (h, w, _) = rgb_image.shape

    # Set image metadata for COCO dataset:
    metadata = {"height" : h,
                    "width" : w,
                    "id" : frame_num,
                    "file_name": image_name}     # Work on image names later
    all_image_info.append(metadata)

    # Run yolov8n on the current image and extract features from resulting boxes
    results = model.predict(rgb_image)
    boxes = results[0].boxes
    assignment, player_center_coord_list, box_coord_list, bbox_annotation = box_to_features(rgb_image, boxes, frame_num)

    # Applies KNN with 3 clusters to find the most prominent colors as in rgb_color
    kmeans = KMeans(n_clusters=3)
    s=kmeans.fit(assignment)
    labels=kmeans.labels_

    # Extract the labels into 2 teams and misc color
    teams = Counter(labels).most_common(3)
    team_1_label = teams[0][0]
    team_2_label = teams[1][0]
    misc_label   = teams[2][0]

    #Return correct labels of (numbers) and the 2 team lab colors
    labels, lab_team_1_color, lab_team_2_color  = misc_in_teams(labels,assignment,teams)

    # Labeling via strings and consistency checks so that team X is always color Y:
    if frame_num == 0:
        global_color_dict["team_1_color"] = lab_team_1_color 
        global_color_dict["team_2_color"] = lab_team_2_color

        labels = list(map(lambda x: x if x != team_1_label else 'Team 1', labels))
        labels = list(map(lambda x: x if x != team_2_label else 'Team 2', labels))
        labels = list(map(lambda x: x if x != misc_label else 'Misc', labels))
        
    if frame_num > 0:
        global_1_vs_local_1 = delta_e_cie2000(global_color_dict["team_1_color"], lab_team_1_color)
        global_1_vs_local_2 = delta_e_cie2000(global_color_dict["team_1_color"], lab_team_2_color)
        if  global_1_vs_local_1 < global_1_vs_local_2:
            #print("Team 1 global is match with team 1 local")
            labels = list(map(lambda x: x if x != team_1_label else 'Team 1', labels))
            labels = list(map(lambda x: x if x != team_2_label else 'Team 2', labels))
        else: 
            #print("Team 1 global is not match with team 1 local")
            labels = list(map(lambda x: x if x != team_2_label else 'Team 1', labels))
            labels = list(map(lambda x: x if x != team_1_label else 'Team 2', labels))
        labels = list(map(lambda x: x if x != misc_label else 'Misc', labels)) 

    # Updated teams indices
    team_1_idx = np.where(np.array(labels) == "Team 1")[0]
    team_2_idx = np.where(np.array(labels) == "Team 2")[0]
    misc_idx   = np.where(np.array(labels) == "Misc")[0] 
    

    # Update the misc color: 
    current_misc_box_coord = [box_coord_list[i] for i in misc_idx]
    current_misc_color = [assignment[i] for i in misc_idx]
    current_misc_lab_colors   = [rgb_to_lab_color(x) for x in current_misc_color]

    current_misc_dict = {'player_center_coord_list': player_center_coord_list,
                          'current_misc_box_coord': current_misc_box_coord,
                          'current_misc_colors': current_misc_color,
                          'current_misc_lab_colors': current_misc_lab_colors,
                          'label': ["Misc"]*len(current_misc_color)}

    
    global_color_dict = update_misc_color(team_1_idx, team_2_idx, misc_idx, 
                                          global_color_dict, current_misc_dict, labels)
    

    #visualize_with_labels(rgb_image, team_1_idx, team_2_idx, misc_idx, labels, global_color_dict, current_misc_dict, box_coord_list)
    final_bbox = update_bbox_label(bbox_annotation,labels)
    for box in final_bbox:
        all_bbox_info.append(box)

    frame_num +=1

#print([x["category_id"] for x in all_bbox_info])
# Write COCO dict
coco_dict["info"] = coco_info
coco_dict["license"] = {'name': vid_name,
                            'id': '',
                            'url': ''}
coco_dict["categories"] = categories_dict
coco_dict["images"] = all_image_info
coco_dict["annotations"] = all_bbox_info.flatten()

with open(os.path.join(destination_path_coco, 'instances_default.json'), 'w') as coco:
    json.dump(coco_dict, coco)

print("Done")


/home/huy-ng/Project/Football-analysis/.venv/lib/python3.8/site-packages/torch/cuda/__init__.py:88: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0

0: 384x640 18 persons, 26.7ms
Speed: 20.1ms preprocess, 26.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)
/home/huy-ng/Project/Football-analysis/.venv/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)

0: 384x640 10 persons, 1 ball, 22.1ms
Speed: 27.2ms preprocess, 22.1ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)
/ho

ValueError: n_samples=1 should be >= n_clusters=3.

In [ ]:
#lab = global_color_dict["team_1_color"]
#rgb = convert_color(lab,sRGBColor)
#rgb.rgb_b

In [ ]:
# def export_coco_dataset_from_prediction(data_path, folder_name, model_name="yolov8n.pt"):
#     # Declare input and output paths
#     model_path = os.path.join(os.path.dirname(os.getcwd()), 'models')
#     print(model_path)
#     model = YOLO(os.path.join(model_path, model_name))

#     images_path = os.path.join(data_path, 'images' + '/' + folder_name)

#     dest_path = os.path.join(data_path, 'coco_datasets' + '/' + folder_name)
#     if not os.path.exists(dest_path):
#         os.mkdir(dest_path)

#     dest_images_path = os.path.join(dest_path, 'images')
#     if not os.path.exists(dest_images_path):
#         os.mkdir(dest_images_path)

#     dest_coco_path = os.path.join(dest_path, 'annotations')
#     if not os.path.exists(dest_coco_path):
#         os.mkdir(dest_coco_path)

#     # Initialize fixed COCO attributes
#     coco_dict ={}
#     frame_id = 1
#     obj_id = 1
#     categories_dict = [{"id" : 1, "name" : "person", "supercategory" : None},
#                     {"id" : 2, "name" : "ball", "supercategory" : None}]

#     creation_date = date.today()
#     year = creation_date.year
#     coco_info = {
#         'contributor': 'Khoa Nguyen, Huy Nguyen',
#         'description': folder_name,
#         'url': '',
#         'version': 0,
#         'date_created': str(creation_date),
#         'year': year,
#     }

#     # Predict and put the information to COCO dict
#     images = []
#     annotations = []

#     for image_name in os.listdir(images_path):
#         frame = os.path.join(images_path, image_name)
#         # Get image shape for COCO dataset:
#         h, w, _ = cv2.imread(frame, cv2.IMREAD_UNCHANGED).shape

#         # Set image metadata for COCO dataset:
#         metadata = {"height" : h,
#                     "width" : w,
#                     "id" : frame_id,
#                     "file_name": image_name}
#         images.append(metadata)

#         # Run YOLOv8 inference on the frame
#         results = model(frame, classes=[0, 32])

#         # Copy original image to coco dataset destination
#         shutil.copy(frame, os.path.join(dest_images_path, image_name))

#         # Write information to coco dataset:
#         boxes = results[0].boxes
#         for box in boxes:
#             (startX, startY, endX, endY) = box.xyxy.cpu().detach().int().tolist()[0]
#             class_id = box.cls[0].item()
#             bbox = {"id" : obj_id,
#                     "image_id" : frame_id,
#                     "category_id" : 1 if class_id == 0.0 else 2,
#                     "segmentation" : [],
#                     "bbox" : [startX, startY, endX-startX, endY-startY],
#                         "area" : (endX-startX) * (endY-startY),
#                     "iscrowd" : 0}
            
#             annotations.append(bbox)
#             obj_id += 1
#         frame_id += 1

#     # Write COCO dict
#     coco_dict["info"] = coco_info
#     coco_dict["license"] = {'name': folder_name,
#                             'id': '',
#                             'url': ''}
#     coco_dict["categories"] = categories_dict
#     coco_dict["images"] = images
#     coco_dict["annotations"] = annotations

#     with open(os.path.join(dest_coco_path, 'instances_default.json'), 'w') as coco:
#         json.dump(coco_dict, coco)

In [ ]:
rgb_image.shape